In [1]:
# =============================================================================
# SHIELD — REPEATED NESTED CV FOR THE 1D CNN ONLY
# =============================================================================
# Same pipeline as the classical classifiers:
#   Outer loop : StratifiedKFold (10 folds) x 5 seeds (42, 0, 1, 7, 99)
#   Inner loop : GridSearchCV over k = {10, 15, 20, 25, 30}
#   Pipeline   : RobustScaler -> SelectKBest(ANOVA F, k) -> CNN
# The outer folds are IDENTICAL to the classical script (same seeds, same Y),
# so CNN predictions can be compared subject-by-subject (McNemar).
# =============================================================================


# %%
# =============================================================================
# CELL 1 — IMPORTS & CONFIGURATION
# =============================================================================
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import time
import pickle

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (StratifiedKFold, LeaveOneOut,
                                     GridSearchCV, train_test_split)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Reshape, Conv1D, BatchNormalization,
                                     MaxPooling1D, Flatten, Dense, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------
# Configuration (same as classical pipeline)
# ------------------------------
DATA_PATH     = "E:/cleaned_csv_reduced_430321.csv"
K_VALUES      = [10, 15, 20, 25, 30]
N_OUTER_FOLDS = 10
OUTER_SEEDS   = [42, 0, 1, 7, 99]
N_BOOTSTRAP   = 1000
SEED          = 42
METRICS = ["accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc"]

# Inner CV for the CNN:
#   StratifiedKFold(5) -> ~1,300 CNN fits in total (about 2 hours)
#   LeaveOneOut()      -> identical to classical, ~22,500 fits (about 35-40 hours)
CNN_INNER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# CNN_INNER_CV = LeaveOneOut()

RESULTS_PATH = "nested_cv_results_cnn_only.pkl"

print(f"TensorFlow {tf.__version__} | inner CV: {CNN_INNER_CV}")


# %%
# =============================================================================
# CELL 2 — DATA LOADING (identical to classical pipeline)
# =============================================================================
df = pd.read_csv(DATA_PATH)

encoder = LabelEncoder()
Y = encoder.fit_transform(df["MDD"]).astype(int)
print("Encoded classes:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

DROP_COLS = ["MDD", "ID", "Species"]
X_df = df.drop(columns=DROP_COLS, errors="ignore")
feature_names = list(X_df.columns)
X = X_df.values.astype(float)

assert X.shape[0] == len(Y)
assert not np.isnan(X).any() and not np.isinf(X).any()
assert len(np.unique(Y)) == 2
print(f"X={X.shape} | MDD={np.sum(Y==1)} | HC={np.sum(Y==0)}")


# %%
# =============================================================================
# CELL 3 — METRIC FUNCTIONS (identical to classical pipeline)
# =============================================================================
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


def bootstrap_ci(y_true, y_pred, y_prob, n_bootstrap=N_BOOTSTRAP):
    results = {m: [] for m in METRICS}
    n = len(y_true)
    for i in range(n_bootstrap):
        idx = resample(np.arange(n), replace=True, stratify=y_true, random_state=i)
        yt, yp, yb = y_true[idx], y_pred[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        results["accuracy"].append(accuracy_score(yt, yp))
        results["precision"].append(precision_score(yt, yp, zero_division=0))
        results["sensitivity"].append(recall_score(yt, yp, zero_division=0))
        results["specificity"].append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        results["f1"].append(f1_score(yt, yp, zero_division=0))
        results["roc_auc"].append(roc_auc_score(yt, yb))
    return {m: (float(np.mean(results[m])),
                float(np.percentile(results[m], 2.5)),
                float(np.percentile(results[m], 97.5))) for m in METRICS}


# %%
# =============================================================================
# CELL 4 — SKLEARN-COMPATIBLE 1D CNN
# =============================================================================
class KerasCNNClassifier(ClassifierMixin, BaseEstimator):

    def __init__(self, filters1=32, filters2=64, kernel_size=3, dense_units=64,
                 dropout=0.3, learning_rate=0.001, epochs=100, batch_size=16,
                 patience=10, val_size=0.1, random_state=SEED):
        self.filters1 = filters1
        self.filters2 = filters2
        self.kernel_size = kernel_size
        self.dense_units = dense_units
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.val_size = val_size
        self.random_state = random_state

    def _build(self, input_dim):
        model = Sequential([
            Input(shape=(input_dim,)),
            Reshape((input_dim, 1)),
            Conv1D(self.filters1, self.kernel_size, activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling1D(2),
            Conv1D(self.filters2, self.kernel_size, activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling1D(2),
            Flatten(),
            Dense(self.dense_units, activation="relu"),
            Dropout(self.dropout),
            Dense(1, activation="sigmoid"),
        ])
        model.compile(optimizer=Adam(learning_rate=self.learning_rate),
                      loss="binary_crossentropy")
        return model

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        class_weight = {0: len(y) / (2 * np.sum(y == 0)),
                        1: len(y) / (2 * np.sum(y == 1))}

        # Stratified early-stopping split from the TRAINING data only
        X_fit, X_val, y_fit, y_val = train_test_split(
            X, y, test_size=self.val_size, stratify=y,
            random_state=self.random_state)

        self.model_ = self._build(X.shape[1])
        es = EarlyStopping(monitor="val_loss", patience=self.patience,
                           restore_best_weights=True, verbose=0)
        hist = self.model_.fit(X_fit, y_fit, validation_data=(X_val, y_val),
                               epochs=self.epochs, batch_size=self.batch_size,
                               class_weight=class_weight, callbacks=[es], verbose=0)
        self.n_epochs_run_ = len(hist.history["loss"])
        return self

    def predict_proba(self, X):
        p = self.model_(np.asarray(X, dtype="float32"), training=False).numpy().ravel()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def make_cnn_grid():
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=K_VALUES[0])),
        ("clf", KerasCNNClassifier(random_state=SEED)),
    ])
    return GridSearchCV(estimator=pipe, param_grid={"sel__k": K_VALUES},
                        cv=CNN_INNER_CV, scoring="accuracy", refit=True,
                        n_jobs=1, error_score=0.0)


# %%
# =============================================================================
# CELL 5 — REPEATED NESTED CV MAIN LOOP (CNN ONLY)
# =============================================================================
all_results = []
t_start = time.time()

for outer_seed in OUTER_SEEDS:
    print(f"\n{'#'*60}\nOUTER SEED = {outer_seed}\n{'#'*60}")
    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True,
                               random_state=outer_seed)

    preds = np.zeros(len(Y), dtype=int)
    probs = np.zeros(len(Y), dtype=float)
    best_k_log, fit_times = [], []

    for fold_idx, (tr, te) in enumerate(outer_cv.split(X, Y), 1):
        gs = make_cnn_grid()
        t0 = time.time()
        gs.fit(X[tr], Y[tr])
        fit_times.append(time.time() - t0)

        best_k = gs.best_params_["sel__k"]
        best_k_log.append(best_k)
        preds[te] = gs.predict(X[te])
        probs[te] = gs.predict_proba(X[te])[:, 1]

        print(f"Fold {fold_idx:02d}/{N_OUTER_FOLDS} | best_k={best_k:<2} | "
              f"time={fit_times[-1]:.1f}s | elapsed={(time.time()-t_start)/60:.1f} min")

    m = compute_metrics(Y, preds, probs)
    all_results.append({
        "outer_seed": outer_seed,
        "preds": preds.copy(),
        "probs": probs.copy(),
        "metrics": m,
        "ci": bootstrap_ci(Y, preds, probs),
        "best_k_log": best_k_log,
        "fit_times": fit_times,
        "Y": Y.copy(),
    })
    with open(RESULTS_PATH, "wb") as f:
        pickle.dump(all_results, f)

    print(f"Seed {outer_seed}: AUC={m['roc_auc']:.3f} | Acc={m['accuracy']*100:.2f}% | "
          f"Sens={m['sensitivity']*100:.2f}% | Spec={m['specificity']*100:.2f}%")

print(f"\nDone in {(time.time()-t_start)/60:.1f} min. Saved to {RESULTS_PATH}")


# %%
# =============================================================================
# CELL 6 — SUMMARY: MEAN ± SD, 95% CI, BEST k, CONFUSION MATRIX
# =============================================================================
print("\n" + "=" * 70)
print("CNN — REPEATED NESTED CV (MEAN ± SD ACROSS 5 SEEDS)")
print("=" * 70)
for m in METRICS:
    v = [r["metrics"][m] for r in all_results]
    print(f"{m:<14}{np.mean(v)*100:8.2f} ± {np.std(v, ddof=1)*100:5.2f} %")

print("\n95% BOOTSTRAP CI (averaged across seeds)")
for m in METRICS:
    means = np.mean([r["ci"][m][0] for r in all_results]) * 100
    lows  = np.mean([r["ci"][m][1] for r in all_results]) * 100
    highs = np.mean([r["ci"][m][2] for r in all_results]) * 100
    print(f"{m:<14}{means:6.2f}% [{lows:6.2f}% — {highs:6.2f}%]")

all_ks = [k for r in all_results for k in r["best_k_log"]]
print("\nBest k distribution:", dict(sorted(Counter(all_ks).items())))

for key in ["TP", "TN", "FP", "FN"]:
    print(f"Mean {key}: {np.mean([r['metrics'][key] for r in all_results]):.1f}")

print(f"\nMean nested fit time per outer fold: "
      f"{np.mean([t for r in all_results for t in r['fit_times']]):.1f} s")


# %%
# =============================================================================
# CELL 7 — OPTIONAL: McNEMAR CNN vs CLASSICAL CLASSIFIERS
# =============================================================================
# Requires the classical predictions saved to a pickle. Add these two lines
# at the END of your classical script (after Cell 7) and re-use the file:
#
#     with open("nested_cv_results_classical.pkl", "wb") as f:
#         pickle.dump(all_nested_results, f)
#
CLASSICAL_PATH = "nested_cv_results_classical.pkl"

if os.path.exists(CLASSICAL_PATH):
    from statsmodels.stats.contingency_tables import mcnemar
    with open(CLASSICAL_PATH, "rb") as f:
        classical = pickle.load(f)

    print("\n" + "=" * 70)
    print("McNEMAR: CNN vs CLASSICAL (per seed, exact test)")
    print("=" * 70)
    for clf in classical[0]["all_preds"].keys():
        pvals, sig = [], 0
        for cnn_r, cls_r in zip(all_results, classical):
            assert cnn_r["outer_seed"] == cls_r["outer_seed"]
            pa, pb = cnn_r["preds"], cls_r["all_preds"][clf]
            b = int(np.sum((pa == Y) & (pb != Y)))
            c = int(np.sum((pa != Y) & (pb == Y)))
            p = mcnemar([[0, b], [c, 0]], exact=True).pvalue
            pvals.append(p)
            sig += int(p < 0.05)
        print(f"CNN vs {clf:<10} mean p={np.mean(pvals):.4f} | "
              f"significant in {sig}/{len(pvals)} seeds")
else:
    print(f"\n{CLASSICAL_PATH} not found — McNemar comparison skipped.")

TensorFlow 2.10.0 | inner CV: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Encoded classes: {0: 0, 1: 1}
X=(91, 314) | MDD=50 | HC=41

############################################################
OUTER SEED = 42
############################################################
Fold 01/10 | best_k=15 | time=124.4s | elapsed=2.1 min
Fold 02/10 | best_k=25 | time=130.2s | elapsed=4.2 min
Fold 03/10 | best_k=15 | time=130.3s | elapsed=6.4 min
Fold 04/10 | best_k=15 | time=126.3s | elapsed=83.9 min
Fold 05/10 | best_k=30 | time=128.7s | elapsed=86.0 min
Fold 07/10 | best_k=10 | time=129.7s | elapsed=90.0 min
Fold 08/10 | best_k=10 | time=131.3s | elapsed=92.2 min
Fold 09/10 | best_k=25 | time=139.4s | elapsed=94.5 min
Fold 10/10 | best_k=15 | time=126.5s | elapsed=96.6 min
Seed 99: AUC=0.562 | Acc=51.65% | Sens=48.00% | Spec=56.10%

Done in 96.8 min. Saved to nested_cv_results_cnn_only.pkl

CNN — REPEATED NESTED CV (MEAN ± SD ACROSS 5 SEEDS)
accuracy         56.48 ±  5.53 %
precisi

In [2]:
# =============================================================================
# SHIELD — COMPUTATIONAL COMPLEXITY OF THE 1D CNN
# Same nested CV pipeline as the CNN performance analysis:
#   Outer : StratifiedKFold (10 folds), seed 42
#           (one repetition, as in the classical complexity analysis)
#   Inner : GridSearchCV + stratified 5-fold CV over k = {10, 15, 20, 25, 30}
#   Pipe  : RobustScaler -> SelectKBest(ANOVA F, k) -> CNN
# Records: training time, inference time per subject, model size,
#          and total nested CV time per outer fold.
# Run this ALONE (no other notebooks running) so timings are accurate.
# =============================================================================

# CELL 1 — IMPORTS & CONFIGURATION
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import time, pickle, tempfile

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Reshape, Conv1D, BatchNormalization,
                                     MaxPooling1D, Flatten, Dense, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

DATA_PATH        = "E:/cleaned_csv_reduced_430321.csv"
K_VALUES         = [10, 15, 20, 25, 30]
N_OUTER_FOLDS    = 10
OUTER_SEEDS      = [42]          # same as the classical complexity analysis
SEED             = 42
N_TIMING_REPEATS = 20            # inference repeated and averaged
CNN_INNER_CV     = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"TensorFlow {tf.__version__} | inner CV: {CNN_INNER_CV}")

# Data loading (identical to main pipeline)
df = pd.read_csv(DATA_PATH)
Y = LabelEncoder().fit_transform(df["MDD"]).astype(int)
X = df.drop(columns=["MDD", "ID", "Species"], errors="ignore").values.astype(float)
print(f"X={X.shape} | MDD={np.sum(Y==1)} | HC={np.sum(Y==0)}")


# =============================================================================
# CELL 2 — SKLEARN-COMPATIBLE 1D CNN (identical to the nested CV script)
# =============================================================================
class KerasCNNClassifier(ClassifierMixin, BaseEstimator):

    def __init__(self, filters1=32, filters2=64, kernel_size=3, dense_units=64,
                 dropout=0.3, learning_rate=0.001, epochs=100, batch_size=16,
                 patience=10, val_size=0.1, random_state=SEED):
        self.filters1 = filters1
        self.filters2 = filters2
        self.kernel_size = kernel_size
        self.dense_units = dense_units
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.val_size = val_size
        self.random_state = random_state

    def _build(self, input_dim):
        model = Sequential([
            Input(shape=(input_dim,)),
            Reshape((input_dim, 1)),
            Conv1D(self.filters1, self.kernel_size, activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling1D(2),
            Conv1D(self.filters2, self.kernel_size, activation="relu", padding="same"),
            BatchNormalization(),
            MaxPooling1D(2),
            Flatten(),
            Dense(self.dense_units, activation="relu"),
            Dropout(self.dropout),
            Dense(1, activation="sigmoid"),
        ])
        model.compile(optimizer=Adam(learning_rate=self.learning_rate),
                      loss="binary_crossentropy")
        return model

    def fit(self, X, y):
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        class_weight = {0: len(y) / (2 * np.sum(y == 0)),
                        1: len(y) / (2 * np.sum(y == 1))}

        # Stratified early-stopping split from the TRAINING data only
        X_fit, X_val, y_fit, y_val = train_test_split(
            X, y, test_size=self.val_size, stratify=y,
            random_state=self.random_state)

        self.model_ = self._build(X.shape[1])
        es = EarlyStopping(monitor="val_loss", patience=self.patience,
                           restore_best_weights=True, verbose=0)
        hist = self.model_.fit(X_fit, y_fit, validation_data=(X_val, y_val),
                               epochs=self.epochs, batch_size=self.batch_size,
                               class_weight=class_weight, callbacks=[es], verbose=0)
        self.n_epochs_run_ = len(hist.history["loss"])
        return self

    def predict_proba(self, X):
        p = self.model_(np.asarray(X, dtype="float32"), training=False).numpy().ravel()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def make_cnn_grid():
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=K_VALUES[0])),
        ("clf", KerasCNNClassifier(random_state=SEED)),
    ])
    return GridSearchCV(estimator=pipe, param_grid={"sel__k": K_VALUES},
                        cv=CNN_INNER_CV, scoring="accuracy", refit=True,
                        n_jobs=1, error_score=0.0)


def cnn_pipeline_size_kb(best_pipe):
    """Size of scaler + selector + CNN weights (comparable to the classical
    pipelines, which include scaler + selector + classifier)."""
    path = tempfile.mktemp(suffix=".pkl")
    try:
        with open(path, "wb") as f:
            pickle.dump({
                "scaler": best_pipe.named_steps["scaler"],
                "sel": best_pipe.named_steps["sel"],
                "cnn_weights": best_pipe.named_steps["clf"].model_.get_weights(),
            }, f)
        return os.path.getsize(path) / 1024
    finally:
        if os.path.exists(path):
            os.unlink(path)


# =============================================================================
# CELL 3 — NESTED CV LOOP: TIMING & SIZE
# =============================================================================
log = {"nested_t": [], "train_t": [], "inf_t": [], "size_kb": [], "best_k": []}
t_start = time.time()

for outer_seed in OUTER_SEEDS:
    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True,
                               random_state=outer_seed)

    for fold_idx, (tr, te) in enumerate(outer_cv.split(X, Y), 1):
        gs = make_cnn_grid()

        # Full nested fit: inner CV over k + refit of the best model
        t0 = time.perf_counter()
        gs.fit(X[tr], Y[tr])
        log["nested_t"].append(time.perf_counter() - t0)

        # Training time of the final selected model (refit on outer training fold)
        log["train_t"].append(gs.refit_time_)
        log["best_k"].append(gs.best_params_["sel__k"])

        # Inference time per subject (final model, averaged over repeats)
        best = gs.best_estimator_
        best.predict(X[te])                       # warm-up call (not timed)
        t0 = time.perf_counter()
        for _ in range(N_TIMING_REPEATS):
            best.predict(X[te])
        log["inf_t"].append((time.perf_counter() - t0) / (N_TIMING_REPEATS * len(te)))

        # Model size
        log["size_kb"].append(cnn_pipeline_size_kb(best))

        print(f"Fold {fold_idx:02d}/{N_OUTER_FOLDS} | best_k={log['best_k'][-1]:<2} | "
              f"train={log['train_t'][-1]:.2f}s | nested={log['nested_t'][-1]:.1f}s | "
              f"elapsed={(time.time()-t_start)/60:.1f} min")

with open("complexity_cnn_nested.pkl", "wb") as f:
    pickle.dump(log, f)


# =============================================================================
# CELL 4 — RESULTS TABLE (same format as the classical table)
# =============================================================================
print("\n" + "=" * 96)
print(f"COMPUTATIONAL COMPLEXITY — mean across {len(OUTER_SEEDS)} seed x {N_OUTER_FOLDS} outer folds")
print("=" * 96)
print(f"{'Classifier':<12}{'Train Time(s)':>16}{'Inference(ms/subj)':>22}"
      f"{'Model Size(KB)':>18}{'Nested CV time(s)':>22}")
print("-" * 96)
print(f"{'CNN':<12}"
      f"{np.mean(log['train_t']):>15.4f}s"
      f"{np.mean(log['inf_t'])*1000:>20.4f}ms"
      f"{np.mean(log['size_kb']):>16.2f}KB"
      f"{np.mean(log['nested_t']):>21.2f}s")
print("=" * 96)
print("Train Time     = refit of the selected model on the outer training fold")
print("Inference      = per subject, final model, averaged over repeats (after warm-up)")
print("Model Size     = scaler + selector + CNN weights")
print("Nested CV time = inner 5-fold CV over k + refit, per outer fold")

TensorFlow 2.10.0 | inner CV: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
X=(91, 314) | MDD=50 | HC=41
Fold 01/10 | best_k=15 | train=4.23s | nested=134.4s | elapsed=2.2 min
Fold 05/10 | best_k=25 | train=4.55s | nested=143.4s | elapsed=12.3 min
Fold 06/10 | best_k=25 | train=6.57s | nested=120.5s | elapsed=14.3 min
Fold 07/10 | best_k=25 | train=3.06s | nested=121.1s | elapsed=16.3 min
Fold 08/10 | best_k=10 | train=2.97s | nested=140.1s | elapsed=18.6 min
Fold 09/10 | best_k=20 | train=3.15s | nested=120.3s | elapsed=20.7 min
Fold 10/10 | best_k=15 | train=6.82s | nested=105.4s | elapsed=22.4 min

COMPUTATIONAL COMPLEXITY — mean across 1 seed x 10 outer folds
Classifier     Train Time(s)    Inference(ms/subj)    Model Size(KB)     Nested CV time(s)
------------------------------------------------------------------------------------------------
CNN                  4.2894s              1.4604ms          104.95KB               134.16s
Train Time     = refit of the select

In [1]:
import xgboost, matplotlib
print(xgboost.__version__, matplotlib.__version__)

2.1.4 3.9.4
